# Reto adicional: estimar precios publicados de Airbnb en Madrid

**Pregunta:** dados atributos publicos de un anuncio, ¿que precio por noche publica? Objetivo: `price`, en euros. No equivale al importe reservado/pagado ni a una recomendacion de precio. El conjunto se actualiza, puede cambiar de columnas y se distribuye con licencia CC BY 4.0; atribuye a Inside Airbnb si reutilizas resultados.

La limpieza convierte precios como `$125.00`, elimina precios imposibles/extremos y coordenadas fuera de Madrid, y registra filas antes/despues. Los identificadores no se usan como predictores.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns
from ml_al_alfa.data import clean_airbnb_data
from ml_al_alfa.train import load_airbnb, train_and_save
from scripts.download_airbnb_madrid import download

sns.set_theme(style="whitegrid")
Path("artifacts").mkdir(exist_ok=True)
source = download()
raw = pd.read_csv(source, low_memory=False)
print("Filas brutas:", len(raw))
print("Duplicados:", int(raw.duplicated().sum()))
display(raw.head())

In [ ]:
X, y, cleaning_report = clean_airbnb_data(raw)
print(cleaning_report.as_dict())
print("Nulos restantes en entrada:", int(X.isna().sum().sum()))
print(f"Rango de precio despues de limpiar: EUR {y.min():.2f} - EUR {y.max():.2f}")
display(X.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=50, ax=axes[0])
axes[0].set(title="Distribucion del precio por noche", xlabel="EUR/noche")
sns.boxplot(x=y, ax=axes[1])
axes[1].set(title="Rango de precios tras filtrar extremos", xlabel="EUR/noche")
plt.tight_layout()

In [ ]:
room_summary = pd.DataFrame({"price": y, "room_type": X["room_type"]}).groupby("room_type")["price"].median().sort_values(ascending=False)
display(room_summary.rename("precio mediano EUR/noche").to_frame())
fig = px.scatter_map(
    X.assign(price=y), lat="latitude", lon="longitude", color="price",
    hover_name="room_type", zoom=10, height=550,
    color_continuous_scale="Plasma", title="Precio publicado por ubicacion en Madrid",
)
fig.show()

Los precios anunciados son muy asimetricos; se filtran valores extremos por encima de 500 EUR/noche para que unos pocos alojamientos no dominen el ajuste. Esta regla es una decision de limpieza transparente, no una conclusion universal sobre el mercado.

In [ ]:
bundle = train_and_save(
    X, y, dataset_name="airbnb", rows=cleaning_report.as_dict()
)
scores = pd.DataFrame(bundle["metrics"]).T.sort_values("mae")
display(scores.style.format({"mae": "{:.2f} EUR/noche", "r2": "{:.4f}"}))
print("Ganador por MAE:", bundle["model_name"])
print("Artefacto guardado en artifacts/airbnb_model.joblib")
if scores.loc[bundle["model_name"], "mae"] < scores.loc["BaselineMedia", "mae"]:
    print("El mejor modelo mejora el baseline en el test aleatorio.")
else:
    print("El mejor modelo no mejora el baseline.")

## Conclusion y limitaciones

El MAE se interpreta en euros por noche. Una particion aleatoria mide generalizacion entre anuncios del mismo corte temporal, no el rendimiento en un futuro distinto. Puede haber dependencia espacial, estacionalidad, cambios de regulacion y atributos incompletos; el objetivo son precios publicados y no transacciones. No usar como tasacion automatica.